In [ ]:
def pair_features(h1, h2):
    return torch.cat([h1, h2, torch.abs(h1 - h2), h1 * h2], dim=-1)

class BranchMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.logit = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        z = self.net(x)
        s = self.logit(z).squeeze(-1)
        return z, s

In [ ]:
class PoolingComparisonModel(nn.Module):
    def __init__(self, d_model=768, hidden_dim=64, dropout=0.4,
                 pooling_type="attention", tau=0.5, topk=3):
        super().__init__()
        self.d_model = d_model
        self.pair_dim = 4 * d_model
        self.pooling_type = pooling_type

        if pooling_type == "mean":
            self.pool = MeanPooling()
        elif pooling_type == "topk":
            self.pool = TopKPooling(k=topk)
        elif pooling_type == "topk_sim":
            self.pool = TopKSimPooling(k=topk)
        elif pooling_type == "attention":
            self.pool = AttentionPooling(tau=tau)
        else:
            raise ValueError(f"Unknown pooling_type: {pooling_type}")

        self.branch_tk = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_ts = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_thk = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_ths = BranchMLP(self.pair_dim, hidden_dim, dropout)

        self.att_w = nn.Linear(hidden_dim, 1)
        self.fuse_out = nn.Linear(hidden_dim, 1)

    def forward(self, batch):
        t = batch["title_emb"].to(DEVICE)
        th = batch["thumb_emb"].to(DEVICE)
        s = batch["stt_embs"].to(DEVICE)
        s_mask = batch["stt_mask"].to(DEVICE)
        k = batch["kf_embs"].to(DEVICE)
        k_mask = batch["kf_mask"].to(DEVICE)

        if self.pooling_type in ["mean", "topk"]:
            s_t, attn_s_t = self.pool(s, s_mask, None)
            k_t, attn_k_t = self.pool(k, k_mask, None)
            s_th, attn_s_th = self.pool(s, s_mask, None)
            k_th, attn_k_th = self.pool(k, k_mask, None)
        else:
            s_t, attn_s_t = self.pool(s, s_mask, t)
            k_t, attn_k_t = self.pool(k, k_mask, t)
            s_th, attn_s_th = self.pool(s, s_mask, th)
            k_th, attn_k_th = self.pool(k, k_mask, th)

        f_tk = pair_features(t, k_t)
        f_ts = pair_features(t, s_t)
        f_thk = pair_features(th, k_th)
        f_ths = pair_features(th, s_th)

        z_tk, s_tk = self.branch_tk(f_tk)
        z_ts, s_ts = self.branch_ts(f_ts)
        z_thk, s_thk = self.branch_thk(f_thk)
        z_ths, s_ths = self.branch_ths(f_ths)

        Z = torch.stack([z_tk, z_ts, z_thk, z_ths], dim=1)   # (B,4,H)

        self.branch_tau = 1.5

        att_scores = self.att_w(Z).squeeze(-1)               # (B,4)
        alpha = F.softmax(att_scores / self.branch_tau, dim=1)

        z_fuse = torch.sum(alpha.unsqueeze(-1) * Z, dim=1)
        s_fuse = self.fuse_out(z_fuse).squeeze(-1)

        return {
            "branch_logits": {
                "tk": s_tk,
                "ts": s_ts,
                "thk": s_thk,
                "ths": s_ths,
            },
            "fused_logit": s_fuse,
            "branch_attention": alpha,
            "token_attention": {
                "s_t": attn_s_t,
                "k_t": attn_k_t,
                "s_th": attn_s_th,
                "k_th": attn_k_th
            }
        }